In [ ]:
# Localiza la raíz del repositorio subiendo desde donde se ejecute el notebook,
# para no depender de una ruta fija de una máquina concreta.
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
while not ((PROJECT_DIR / "data").exists() and (PROJECT_DIR / "notebooks").exists()):
    PROJECT_DIR = PROJECT_DIR.parent

# 1. Importación

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv(f"{PROJECT_DIR}/data/processed/euskadi/listings_full_features.csv")

## 2. Preparación de X e y

### 2.1 Identificadores, objetivo y features

**Fuga corregida más abajo**: `neighbourhood_price_encoded` (creada en `02_feature_engineering.ipynb`) se calculó allí usando todo el dataset, no solo lo que aquí es train. Se arregla en la sección 3.1bis, justo después del split. Por eso `neighbourhood_cleansed` (la columna cruda) sigue en `df` en este punto.

In [3]:
id_cols = ["id", "host_id", "host_profile_id"]
target_col = "price"
leakage_cols = ["price_log", "price_per_accommodate", "price_per_min_night"]
# neighbourhood_cleansed todavía no es una feature (pendiente de la corrección de la
# sección 3.1bis): se excluye de X igual que los identificadores, pero se mantiene en
# df para poder usarla justo después del split.
pending_cols = ["neighbourhood_cleansed"]
feature_cols = [c for c in df.columns if c not in id_cols + [target_col] + leakage_cols + pending_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

X.shape, y.shape

((5670, 71), (5670,))

### 2.2 Booleanas a 0/1

In [4]:
bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)
len(bool_cols)

40

### 2.3 Nulos restantes

In [5]:
null_cols = X.columns[X.isnull().any()].tolist()
print(null_cols)

X[null_cols] = X[null_cols].fillna(X[null_cols].median())
X.isnull().sum().sum()

['review_scores_rating', 'listing_age_days', 'days_since_last_review']


np.int64(0)

`X` queda con 71 columnas numéricas sin nulos, e `y` es `price` sin transformar (el logaritmo se aplica más adelante, solo para el modelo de la sección 6).

## 3. Train/test split

### 3.1 Dividir

In [6]:
room_type_cols = [c for c in df.columns if c.startswith("room_type_")]
room_type_for_stratify = df[room_type_cols].idxmax(axis=1)

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.2, random_state=42, stratify=room_type_for_stratify
)

X_train.shape, X_test.shape

((4536, 71), (1134, 71))

Estratificado por `room_type` (4164 `Entire home/apt`, 1417 `Private room`, 47 `Hotel room`, 42 `Shared room`) para que las dos categorías minoritarias queden representadas en ambos conjuntos pese a su tamaño.

### 3.1bis Corregir la fuga de `neighbourhood_price_encoded`

In [7]:
smoothing = 10
global_mean_price_train = y_train.mean()
neigh_stats_train = df_train.groupby("neighbourhood_cleansed")["price"].agg(["mean", "count"])
smoothed_mean_train = (
    neigh_stats_train["count"] * neigh_stats_train["mean"] + smoothing * global_mean_price_train
) / (neigh_stats_train["count"] + smoothing)

df_train["neighbourhood_price_encoded"] = df_train["neighbourhood_cleansed"].map(smoothed_mean_train)
df_test["neighbourhood_price_encoded"] = (
    df_test["neighbourhood_cleansed"].map(smoothed_mean_train).fillna(global_mean_price_train)
)

print("municipios de test no vistos en train:", df_test["neighbourhood_cleansed"].map(smoothed_mean_train).isnull().sum())

# X_train/X_test ya tenían la versión con fuga (calculada en la sección 2.1 antes del
# split): se sincronizan con el valor corregido de df_train/df_test.
X_train["neighbourhood_price_encoded"] = df_train["neighbourhood_price_encoded"]
X_test["neighbourhood_price_encoded"] = df_test["neighbourhood_price_encoded"]

df_train = df_train.drop(columns=["neighbourhood_cleansed"])
df_test = df_test.drop(columns=["neighbourhood_cleansed"])

df_train[["neighbourhood_price_encoded"]].describe()

municipios de test no vistos en train: 8


,neighbourhood_price_encoded
count,4536.000000
mean,269.328985
std,76.753507
min,161.545936
25%,211.782427
50%,236.718049
75%,349.479779
max,574.920540


8 filas de test (repartidas en **6 de los 206 municipios** del dataset) quedan sin equivalente en train: `Arrazua-Ubarrundia` y `Harana` aportan 2 filas cada uno, y `Elvillar`, `Orexa`, `Zerain` y `Zizurkil` 1 cada uno: municipios con 1-2 anuncios en todo el dataset, así que el reparto 80/20 los deja fuera de train por puro azar. Para esas 8 filas, `neighbourhood_price_encoded` en test cae de vuelta a la media global de train, el respaldo para el que se diseñó el `.fillna()`.

Nota aparte: el dataset tiene **206 municipios**, no los 207 de la EDA: `Iruña Oka` tenía un único anuncio con `price` nulo, así que desapareció por completo en la limpieza de la sección 4 de `01_eda.ipynb`, antes de llegar aquí.

### 3.2 Guardar el split

In [8]:
df_train.to_csv(f"{PROJECT_DIR}/data/processed/euskadi/listings_train.csv", index=False)
df_test.to_csv(f"{PROJECT_DIR}/data/processed/euskadi/listings_test.csv", index=False)

## 4. Baseline ingenuo

### 4.1 Predecir siempre la media

In [9]:
global_mean = y_train.mean()
pred_mean = np.full(len(y_test), global_mean)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_mean)))
print("MAE:", mean_absolute_error(y_test, pred_mean))
print("R2:", r2_score(y_test, pred_mean))

RMSE: 253.3001891191594
MAE: 151.53578996481997
R2: -0.0004600960954723732


### 4.2 Predecir siempre la mediana

In [10]:
global_median = y_train.median()
pred_median = np.full(len(y_test), global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_median)))
print("MAE:", mean_absolute_error(y_test, pred_median))
print("R2:", r2_score(y_test, pred_median))

RMSE: 261.24573433105627
MAE: 136.00134038800704
R2: -0.06420956751925044


El MAE mejora (136.00€ frente a 151.54€ con la media), pero el R² empeora ligeramente (-0.06 frente a ~0.00): el mismo patrón visto en el resto de ciudades, esperable dada la asimetría de `price`.

### 4.3 Predecir la mediana según `accommodates`

In [11]:
train_medians_by_accommodates = X_train.assign(price=y_train).groupby("accommodates")["price"].median()

pred_accommodates = X_test["accommodates"].map(train_medians_by_accommodates).fillna(global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_accommodates)))
print("MAE:", mean_absolute_error(y_test, pred_accommodates))
print("R2:", r2_score(y_test, pred_accommodates))

RMSE: 229.26069911343419
MAE: 111.13467372134038
R2: 0.18042638587759918


Mejora clara en las cuatro métricas a la vez (MAE 136.00€→111.13€, R² -0.06→0.18): a diferencia de Sevilla, aquí `accommodates` por sí solo ya reduce el error de forma sustancial.

## 5. Métricas de evaluación

### 5.1 Función `evaluate`

In [12]:
def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"modelo": name, "RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

### 5.2 Tabla comparativa de los baselines

In [13]:
rows = [
    evaluate(y_test, pred_mean, "Media"),
    evaluate(y_test, pred_median, "Mediana"),
    evaluate(y_test, pred_accommodates, "Mediana por accommodates"),
]

results = pd.DataFrame(rows).set_index("modelo")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,253.30,151.54,-0.00,99.73
Mediana,261.25,136.00,-0.06,68.31
Mediana por accommodates,229.26,111.13,0.18,45.57


El MAPE mejora con claridad en cada paso (99.73% → 68.31% → 45.57%), la misma dirección que en el resto de ciudades.

## 6. Baseline real: regresión lineal

### 6.1 Entrenar

In [14]:
y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train, y_train_log)

pred_log = model.predict(X_test)
pred_lr = np.expm1(pred_log)

/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


### 6.2 Evaluar

In [15]:
results.loc["Regresión lineal (log)"] = evaluate(y_test, pred_lr, "Regresión lineal (log)")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,253.30,151.54,-0.00,99.73
Mediana,261.25,136.00,-0.06,68.31
Mediana por accommodates,229.26,111.13,0.18,45.57
Regresión lineal (log),183.65,79.57,0.47,32.40


**R²=0.47, MAE≈79.57€, MAPE≈32.4%**: a diferencia de Sevilla (R²=0.06, el caso más extremo del proyecto), aquí las cuatro métricas mejoran de forma limpia y monótona en cada paso del baseline (RMSE 253.30€→261.25€→229.26€→183.65€, MAE 151.54€→136.00€→111.13€→79.57€, R² -0.00→-0.06→0.18→0.47, MAPE 99.73%→68.31%→45.57%→32.40%). Coherente con las correlaciones de Spearman ya vistas en la EDA/FE (`accommodates` 0.58, `bedrooms` 0.55): aquí la relación tamaño-precio es lo bastante fuerte como para que ni siquiera un modelo lineal simple se vea dominado por la cola alta de `price`.

In [16]:
y_test.describe()

count    1134.000000
mean      260.310511
std       253.353671
min         6.790000
25%       127.372500
50%       195.335000
75%       305.875000
max      4222.000000
Name: price, dtype: float64

`y_test` tiene una `std` de ~253€ sobre una mediana de ~195€: una cola larga real, pero menos extrema que la de Sevilla (`std` ~299€ sobre mediana ~110€, una proporción mucho mayor).

In [17]:
above_1000 = y_test[y_test > 1000]
print("filas por encima de 1000€:", len(above_1000), "de", len(y_test))
print("std sin esas filas:", y_test[y_test <= 1000].std())

filas por encima de 1000€: 16 de 1134
std sin esas filas: 171.63684776001926


16 filas por encima de 1000€ (1.4% del test) explican parte de la varianza: reducen la `std` de 253€ a 172€ si se excluyen, un efecto real pero mucho más repartido que en Sevilla (donde 6 filas, el 0.4%, más que duplicaban la `std`). Consistente con que el R² aquí no se derrumba: el modelo sí logra anticipar precios altos en anuncios grandes (varias de estas filas son precisamente las villas/viviendas grandes ya vistas en la EDA, incluida la propia fila de la final de la Europa League).

### 6.3 Un aviso a tener en cuenta

In [18]:
import numpy.linalg as la

la.cond(X_train.values)

np.float64(2.6613874374166508e+19)

Número de condición **2.66×10¹⁹**, incluso más alto que el ya elevado de Mallorca (2.44×10¹⁹): el más alto visto hasta ahora en el proyecto. Misma causa que en el resto de ciudades: `X` incluye a la vez la versión bruta y la versión `_log` de varias variables, más varias codificaciones categóricas solapadas. No invalida las métricas (`scikit-learn` resuelve con SVD, sin `NaN`/`inf` en las predicciones, aunque sí lanza un `RuntimeWarning` de overflow interno, benigno), pero impide interpretar los coeficientes uno a uno. Se deja igual que en las otras ciudades para `04_model_training.ipynb`.

## 7. Conclusiones

Resumen de los resultados del baseline.

### Resultados

| Modelo | RMSE | MAE | R² | MAPE |
|---|---|---|---|---|
| Media | 253.30 | 151.54 | -0.00 | 99.73 |
| Mediana | 261.25 | 136.00 | -0.06 | 68.31 |
| Mediana por `accommodates` | 229.26 | 111.13 | 0.18 | 45.57 |
| Regresión lineal (log) | 183.65 | 79.57 | 0.47 | 32.40 |

Las cuatro métricas mejoran de forma limpia y monótona en cada paso, a diferencia de Sevilla, donde RMSE/R² apenas se movían pese a que MAE/MAPE sí mejoraban. Aquí no hace falta ninguna explicación adicional: el baseline se comporta como cabría esperar.

### Tres problemas encontrados y cómo se trataron

- **Fuga de información directa**: `price_log`, `price_per_accommodate` y `price_per_min_night` estaban calculadas a partir de `price` y se habían colado como features. Se excluyeron en la sección 2, igual que en las otras cinco ciudades.
- **Fuga más leve, ahora corregida**: `neighbourhood_price_encoded` se calculaba con todo el dataset. Se corrige en la sección 3.1bis, recalculándola solo con `df_train`, con la particularidad de que 6 de los 206 municipios (8 filas de test) no aparecen en train y caen de vuelta a la media global.
- **Un municipio desaparecido antes de llegar aquí**: `Iruña Oka` tenía un único anuncio con `price` nulo, descartado ya en la EDA: de ahí que el dataset tenga 206 municipios y no 207.
- **Multicolinealidad severa** en la regresión lineal, por la versión bruta y `_log` de varias variables a la vez (número de condición 2.66×10¹⁹, el más alto del proyecto hasta ahora). No afecta a las métricas de predicción, pero impide interpretar los coeficientes uno a uno.

### Una diferencia real con Sevilla, no un error de pipeline

El R² del baseline lineal en Euskadi (0.47) está muy por encima del de Sevilla (0.06) y en línea con el resto de ciudades del proyecto. La razón no es ningún ajuste distinto en el pipeline (es exactamente el mismo código y las mismas reglas), sino una diferencia real de los datos: la cola alta de `price` en Euskadi está mucho más ligada al tamaño del anuncio (villas grandes, la propia fila de la final de la Europa League) que en Sevilla, donde varios precios extremos de la cola alta no guardaban relación clara con `accommodates`/`bedrooms`. Por eso aquí RMSE y R² mejoran en la misma dirección que MAE y MAPE, sin la contradicción aparente que hubo que investigar en Sevilla.

### El listón para `04_model_training.ipynb`

Cualquier modelo más complejo (Random Forest, Extra Trees, HistGradientBoosting...) tiene que superar **RMSE≈183.65€ / MAE≈79.57€ / R²≈0.47 / MAPE≈32.4%** como referencia. Un modelo de árboles no necesita la imputación por mediana de la sección 2.3, no le afecta la multicolinealidad de la sección 6.3, y debería seguir mejorando sobre todo aprovechando mejor las interacciones entre tamaño, `neighbourhood_price_encoded` y `distance_to_center_km` (con su patrón heterogéneo por capital, sección 8.1 de la FE) que una regresión lineal no puede capturar.

### Lo que queda guardado

`listings_train.csv` y `listings_test.csv` en `data/processed/euskadi/`, con el mismo split (80/20, `random_state=42`, estratificado por `room_type`) y `neighbourhood_price_encoded` ya corregido, para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` trabajen sobre las mismas filas y los resultados sean comparables entre notebooks.